<a href="https://colab.research.google.com/github/vmyel/thesis_ref/blob/main/pd_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# 0.  Mount Google Drive & install/import libraries
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# 1.  USER-DEFINED PATHS  –– adjust to your Drive layout
# ============================================================
METADATA_PATH = '/content/drive/MyDrive/teasis/PaHaW/PaHaW_files/corpus_PaHaW.xlsx'
SVC_ROOT      = '/content/drive/MyDrive/teasis/PaHaW/PaHaW_public/'   # folder that contains 00008/, 00009/, … sub-folders

# ============================================================
# 2.  Load & inspect metadata
# ============================================================
meta = pd.read_excel(METADATA_PATH, dtype={'ID': str})

# Normalise column names (strip whitespace, consistent case)
meta.columns = meta.columns.str.strip()

# Make sure the subject ID column is zero-padded to 5 chars
# (adjust 'ID' to whatever the actual column header is)
meta['ID'] = meta['ID'].astype(str).str.zfill(5)

print("Metadata shape :", meta.shape)
print("Columns        :", meta.columns.tolist())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Metadata shape : (75, 11)
Columns        : ['ID', 'Nationality', 'Sex', 'Disease', 'PD status', 'Age', 'Dominant hand', 'LED', 'UPDRS V', 'Length of PD', 'Unnamed: 10']


In [3]:
# ============================================================
# 3.  Filter out SEVERE stages  (H&Y / UPDRS V  >= 4.0)
#     NaN means Healthy Control → must be preserved
# ============================================================
before = len(meta)

# Keep row if UPDRS V is NaN (HC) OR score is below 4.0
meta_filtered = meta[meta['UPDRS V'].isna() | (meta['UPDRS V'] < 4.0)].copy()

after = len(meta_filtered)

print(f"Removed {before - after} subject(s) with UPDRS V >= 4.0")
print(f"Remaining subjects: {after}")   # should now be ~72

# Quick sanity check
print("\nUPDRS V distribution after filter:")
print(meta_filtered['UPDRS V'].value_counts(dropna=False).sort_index())

Removed 3 subject(s) with UPDRS V >= 4.0
Remaining subjects: 72

UPDRS V distribution after filter:
UPDRS V
1.0     5
2.0    18
2.5     6
3.0     5
NaN    38
Name: count, dtype: int64


In [4]:
# ============================================================
# 4.  Assign group labels
#       • Healthy Controls : UPDRS V is NaN  (no H&Y score)
#       • Early PD         : UPDRS V  1.0 – 2.0
#       • Moderate PD      : UPDRS V  2.5 – 3.0
# ============================================================
def assign_group(row):
    score = row['UPDRS V']

    # Healthy controls have no UPDRS V score → NaN
    if pd.isna(score):
        return 'Healthy Control'
    elif 1.0 <= score <= 2.0:
        return 'Early PD'
    elif 2.5 <= score <= 3.0:
        return 'Moderate PD'
    else:
        return 'Unknown'   # safety catch

meta_filtered['Group'] = meta_filtered.apply(assign_group, axis=1)

print("Group distribution:")
print(meta_filtered['Group'].value_counts())
print()
print(meta_filtered[['ID', 'Disease', 'UPDRS V', 'Group']].head(72).to_string(index=False))

Group distribution:
Group
Healthy Control    38
Early PD           23
Moderate PD        11
Name: count, dtype: int64

   ID Disease  UPDRS V           Group
00008      PD      1.0        Early PD
00009      PD      1.0        Early PD
00013      PD      1.0        Early PD
00043      PD      1.0        Early PD
00044      PD      1.0        Early PD
00001      PD      2.0        Early PD
00002      PD      2.0        Early PD
00003      PD      2.0        Early PD
00004      PD      2.0        Early PD
00005      PD      2.0        Early PD
00006      PD      2.0        Early PD
00016      PD      2.0        Early PD
00019      PD      2.0        Early PD
00020      PD      2.0        Early PD
00022      PD      2.0        Early PD
00023      PD      2.0        Early PD
00033      PD      2.0        Early PD
00034      PD      2.0        Early PD
00036      PD      2.0        Early PD
00053      PD      2.0        Early PD
00054      PD      2.0        Early PD
00078      PD      2.0 

In [5]:
# ============================================================
# 5.  Helper – parse a single SVC file
# ============================================================
def parse_svc(filepath: str) -> pd.DataFrame:
    """
    SVC format
    ----------
    Line 1   : number of samples (integer)
    Lines 2+ : Y  X  timestamp  button_state  azimuth  altitude  pressure
    """
    with open(filepath, 'r') as fh:
        lines = fh.read().splitlines()

    n_samples = int(lines[0].strip())
    data_lines = lines[1: n_samples + 1]          # safety slice

    records = []
    for line in data_lines:
        parts = line.split()
        if len(parts) < 7:
            continue
        records.append({
            'y'            : float(parts[0]),
            'x'            : float(parts[1]),
            'timestamp'    : float(parts[2]),
            'button_state' : int(parts[3]),
            'azimuth'      : float(parts[4]),
            'altitude'     : float(parts[5]),
            'pressure'     : float(parts[6]),
        })

    df = pd.DataFrame(records)
    df['n_declared'] = n_samples
    return df

In [6]:
# ============================================================
# 6.  Load ALL SVC files  (task number only, no session)
#     Filename format:  {subjectID}_{task}_{repetition}
#     e.g.  00098_6_1  →  subject=00098, task=6
# ============================================================
all_records = []
valid_ids   = set(meta_filtered['ID'].tolist())

for subj_folder in sorted(Path(SVC_ROOT).iterdir()):
    if not subj_folder.is_dir():
        continue

    subj_id = subj_folder.name.zfill(5)   # e.g. '98' → '00098'

    if subj_id not in valid_ids:
        continue

    meta_row = meta_filtered[meta_filtered['ID'] == subj_id].iloc[0]

    svc_files = sorted([f for f in subj_folder.iterdir() if f.is_file()])

    for svc_path in svc_files:
        try:
            svc_df = parse_svc(str(svc_path))
        except Exception as e:
            print(f"  [WARN] Could not parse {svc_path.name}: {e}")
            continue

        # e.g. '00098_6_1' → parts = ['00098', '6', '1']
        parts = svc_path.stem.split('_')
        task  = parts[1] if len(parts) > 1 else 'unknown'
        # parts[2] is just the repetition suffix ('1'), not used

        svc_df['subject_id'] = subj_id
        svc_df['task']       = task          # task number (1-8)
        svc_df['file_name']  = svc_path.name

        # Attach metadata
        svc_df['group']      = meta_row['Group']
        svc_df['updrs_v']    = meta_row['UPDRS V']
        svc_df['disease']    = meta_row['Disease']
        svc_df['age']        = meta_row['Age']
        svc_df['sex']        = meta_row['Sex']

        all_records.append(svc_df)

full_df = pd.concat(all_records, ignore_index=True)

print(f"Total stroke-point rows : {len(full_df):,}")
print(f"Unique subjects         : {full_df['subject_id'].nunique()}")
print(f"Unique SVC files        : {full_df['file_name'].nunique()}")
# print(f"\nTasks found             : {sorted(full_df['task'].unique())}")

Total stroke-point rows : 1,303,790
Unique subjects         : 72
Unique SVC files        : 573


In [7]:
# ============================================================
# 7.  Split into the three group DataFrames
# ============================================================
df_healthy  = full_df[full_df['group'] == 'Healthy Control'].copy()
df_early    = full_df[full_df['group'] == 'Early PD'].copy()
df_moderate = full_df[full_df['group'] == 'Moderate PD'].copy()

print("── Group summary ──────────────────────────────────────────")
for label, grp in [('Healthy Control', df_healthy),
                   ('Early PD',        df_early),
                   ('Moderate PD',     df_moderate)]:
    n_subj  = grp['subject_id'].nunique()
    n_files = grp['file_name'].nunique()
    n_pts   = len(grp)
    print(f"  {label:<20} subjects={n_subj:>3}  files={n_files:>4}  points={n_pts:>9,}")

── Group summary ──────────────────────────────────────────
  Healthy Control      subjects= 38  files= 302  points=  636,178
  Early PD             subjects= 23  files= 184  points=  427,783
  Moderate PD          subjects= 11  files=  87  points=  239,829


PREPROCESSING

In [8]:
# ============================================================
# 8.  IMPORTS FOR PREPROCESSING & CROSS-VALIDATION
# ============================================================
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from scipy.interpolate import interp1d
from collections import OrderedDict

# ============================================================
# 9.  DEFINE CLASSIFICATION TASKS
# ============================================================
# Task 1 – Detection:  Healthy Control vs PD (Early + Moderate)
# Task 2 – Staging:    Early PD vs Moderate PD

def get_detection_label(group):
    """Binary: 0 = Healthy Control, 1 = PD (Early or Moderate)."""
    return 0 if group == 'Healthy Control' else 1

def get_staging_label(group):
    """Binary: 0 = Early PD, 1 = Moderate PD."""
    if group == 'Early PD':
        return 0
    elif group == 'Moderate PD':
        return 1
    else:
        return None  # HC excluded from staging

# ============================================================
# 10.  BUILD PATIENT-LEVEL TABLE FOR STRATIFICATION
# ============================================================
patient_table = (
    full_df
    .groupby('subject_id')
    .agg(group=('group', 'first'))
    .reset_index()
)

# Add labels for both tasks
patient_table['label_detect'] = patient_table['group'].apply(get_detection_label)
patient_table['label_stage']  = patient_table['group'].apply(get_staging_label)

print("Patient table (all):")
print(patient_table['group'].value_counts())
print(f"\nTotal patients: {len(patient_table)}")

# For staging, only PD patients
patient_table_staging = patient_table[patient_table['label_stage'].notna()].copy()
patient_table_staging['label_stage'] = patient_table_staging['label_stage'].astype(int)

print(f"\nStaging patients: {len(patient_table_staging)}")
print(patient_table_staging['group'].value_counts())

# ============================================================
# 11.  COMPUTE AVERAGE SEQUENCE LENGTH ACROSS ALL DRAWINGS
# ============================================================

# Count the number of time-steps (rows) per drawing
seq_lengths = (
    full_df
    .groupby(['subject_id', 'file_name'])
    .size()
    .reset_index(name='seq_len')
)

avg_seq_len = seq_lengths['seq_len'].mean()
SEQ_LEN     = int(round(avg_seq_len))

print(f"\n── Sequence Length Statistics ──────────────────────────────")
print(f"  Total drawings     : {len(seq_lengths)}")
print(f"  Min  seq length    : {seq_lengths['seq_len'].min()}")
print(f"  Max  seq length    : {seq_lengths['seq_len'].max()}")
print(f"  Mean seq length    : {avg_seq_len:.1f}")
print(f"  Median seq length  : {seq_lengths['seq_len'].median():.1f}")
print(f"  Std  seq length    : {seq_lengths['seq_len'].std():.1f}")
print(f"  ► SEQ_LEN (rounded): {SEQ_LEN}")

# ============================================================
# 12.  CONSTANTS
# ============================================================
N_FOLDS      = 5
RANDOM_STATE = 42

# Deep-learning feature columns (raw time-series)
DL_FEATURES = ['x', 'y', 'pressure', 'button_state']   # button_state encodes in-air

# ============================================================
# 13.  PATH A — DEEP LEARNING PREPROCESSING FUNCTIONS
# ============================================================

def select_dl_features(df: pd.DataFrame) -> pd.DataFrame:
    """Select the 4 channels for DL: x, y, pressure, in-air (button_state)."""
    return df[DL_FEATURES].copy()


def pad_or_clip_sequence(arr: np.ndarray, target_len: int = SEQ_LEN) -> np.ndarray:
    """
    Adjust a 2-D array (time_steps, n_features) to exactly `target_len` steps.
    - If longer  → clip (truncate) to the first `target_len` steps.
    - If shorter → zero-pad at the end.
    """
    n_steps, n_feat = arr.shape
    if n_steps >= target_len:
        return arr[:target_len, :]
    else:
        padded = np.zeros((target_len, n_feat), dtype=arr.dtype)
        padded[:n_steps, :] = arr
        return padded


def build_dl_sequences(df: pd.DataFrame, target_len: int = SEQ_LEN):
    """
    Convert a full DataFrame (multiple files/drawings) into
    a list of fixed-length sequences grouped by (subject_id, file_name).

    Returns
    -------
    sequences : np.ndarray  — shape (n_samples, target_len, n_features)
    subjects  : list         — subject_id per sequence (for later grouping)
    filenames : list         — file_name per sequence
    """
    sequences = []
    subjects  = []
    filenames = []

    for (subj, fname), grp in df.groupby(['subject_id', 'file_name']):
        feat = select_dl_features(grp).values.astype(np.float32)
        seq  = pad_or_clip_sequence(feat, target_len)
        sequences.append(seq)
        subjects.append(subj)
        filenames.append(fname)

    return np.array(sequences), subjects, filenames


# ============================================================
# 14.  PATH B — MACHINE LEARNING FEATURE ENGINEERING
# ============================================================

def compute_kinematics(grp: pd.DataFrame) -> dict:
    """
    Compute kinematic and hesitation features from a single drawing (SVC file).

    Returns a flat dictionary of scalar features.
    """
    features = OrderedDict()

    x   = grp['x'].values.astype(np.float64)
    y   = grp['y'].values.astype(np.float64)
    t   = grp['timestamp'].values.astype(np.float64)
    p   = grp['pressure'].values.astype(np.float64)
    btn = grp['button_state'].values.astype(np.int32)

    n = len(x)

    # ── Time deltas (avoid division by zero) ────────────────
    dt = np.diff(t)
    dt[dt == 0] = 1e-6  # replace zero intervals

    # ── Displacement ────────────────────────────────────────
    dx = np.diff(x)
    dy = np.diff(y)
    ds = np.sqrt(dx**2 + dy**2)

    # ── Velocity ────────────────────────────────────────────
    vx = dx / dt
    vy = dy / dt
    velocity = ds / dt

    # ── Acceleration ────────────────────────────────────────
    if len(velocity) >= 2:
        dt2 = dt[:-1]
        dt2[dt2 == 0] = 1e-6
        dv = np.diff(velocity)
        acceleration = dv / dt2
    else:
        acceleration = np.array([0.0])

    # ── Jerk ────────────────────────────────────────────────
    if len(acceleration) >= 2:
        dt3 = dt[:-2] if len(dt) >= 3 else np.array([1e-6])
        dt3[dt3 == 0] = 1e-6
        da = np.diff(acceleration)
        jerk = da / dt3[:len(da)]
    else:
        jerk = np.array([0.0])

    # ── Statistical functionals for velocity ────────────────
    features['velocity_mean']   = np.mean(velocity)
    features['velocity_median'] = np.median(velocity)
    features['velocity_std']    = np.std(velocity)
    features['velocity_max']    = np.max(velocity)

    # ── Statistical functionals for acceleration ────────────
    features['accel_mean']   = np.mean(acceleration)
    features['accel_median'] = np.median(acceleration)
    features['accel_std']    = np.std(acceleration)
    features['accel_max']    = np.max(np.abs(acceleration))

    # ── Statistical functionals for jerk ────────────────────
    features['jerk_mean']   = np.mean(jerk)
    features['jerk_median'] = np.median(jerk)
    features['jerk_std']    = np.std(jerk)
    features['jerk_max']    = np.max(np.abs(jerk))

    # ── Pressure profile ────────────────────────────────────
    on_surface_mask = (btn == 1)
    p_on = p[on_surface_mask] if on_surface_mask.any() else np.array([0.0])

    features['pressure_mean']   = np.mean(p_on)
    features['pressure_median'] = np.median(p_on)
    features['pressure_std']    = np.std(p_on)
    features['pressure_max']    = np.max(p_on)

    # ── Motor hesitation / spatio-temporal metrics ──────────

    # In-air vs on-surface time
    in_air_mask     = (btn == 0)
    on_surface_mask = (btn == 1)

    # Total durations (sum of dt where corresponding sample is in-air / on-surface)
    btn_intervals = btn[:-1]  # align with dt
    dt_in_air     = dt[btn_intervals == 0]
    dt_on_surface = dt[btn_intervals == 1]

    features['total_in_air_time']     = np.sum(dt_in_air)
    features['total_on_surface_time'] = np.sum(dt_on_surface)
    features['total_time']            = t[-1] - t[0] if n > 1 else 0.0

    # Ratio
    total = features['total_in_air_time'] + features['total_on_surface_time']
    features['in_air_time_ratio'] = (
        features['total_in_air_time'] / total if total > 0 else 0.0
    )

    # Stroke count (number of pen-down segments)
    transitions = np.diff(btn)
    features['stroke_count']   = int(np.sum(transitions == 1))
    features['pen_lift_count'] = int(np.sum(transitions == -1))

    # Path length (total displacement)
    features['path_length'] = np.sum(ds)

    # Number of samples
    features['n_samples'] = n

    return features


def build_ml_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transform the full time-series DataFrame into a feature matrix
    with one row per drawing (subject_id × file_name).
    """
    rows = []
    for (subj, fname), grp in df.groupby(['subject_id', 'file_name']):
        feat = compute_kinematics(grp)
        feat['subject_id'] = subj
        feat['file_name']  = fname
        feat['task']       = grp['task'].iloc[0]
        feat['group']      = grp['group'].iloc[0]
        rows.append(feat)

    feat_df = pd.DataFrame(rows)
    return feat_df


# ============================================================
# 15.  BUILD FULL FEATURE SETS (before fold splitting)
# ============================================================

# --- Path A: DL sequences ---
print(f"\nBuilding DL sequences (SEQ_LEN = {SEQ_LEN}) …")
dl_sequences, dl_subjects, dl_filenames = build_dl_sequences(full_df, target_len=SEQ_LEN)
print(f"  DL tensor shape : {dl_sequences.shape}")   # (n_drawings, SEQ_LEN, 4)

# --- Path B: ML feature vectors ---
print("Building ML feature vectors …")
ml_features = build_ml_feature_matrix(full_df)

# Identify numeric feature columns (exclude identifiers)
ml_id_cols   = ['subject_id', 'file_name', 'task', 'group']
ml_feat_cols = [c for c in ml_features.columns if c not in ml_id_cols]

print(f"  ML feature matrix : {ml_features.shape}")
print(f"  Feature columns   : {len(ml_feat_cols)}")
print(f"  Features: {ml_feat_cols}")


# ============================================================
# 16.  PATIENT-LEVEL STRATIFIED 5-FOLD CROSS-VALIDATION
#      WITH IN-FOLD PREPROCESSING
# ============================================================

def run_cv_preprocessing(task_name: str,
                         patient_df: pd.DataFrame,
                         label_col: str,
                         full_data: pd.DataFrame,
                         dl_seqs: np.ndarray,
                         dl_subjs: list,
                         ml_feat_df: pd.DataFrame,
                         ml_feat_cols: list,
                         n_folds: int = N_FOLDS):
    """
    Execute patient-level stratified K-Fold.
    For each fold:
      - Split patients into train / test
      - PATH A: fit Z-score scaler on train DL data, transform both
      - PATH B: fit Z-score scaler on train ML vectors, transform both

    Returns a list of fold dictionaries containing preprocessed data.
    """
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)

    subjects = patient_df['subject_id'].values
    labels   = patient_df[label_col].values

    folds = []

    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(subjects, labels)):
        print(f"\n{'='*60}")
        print(f"  {task_name} — Fold {fold_idx + 1}/{n_folds}")
        print(f"{'='*60}")

        train_subjects = set(subjects[train_idx])
        test_subjects  = set(subjects[test_idx])

        train_labels_fold = labels[train_idx]
        test_labels_fold  = labels[test_idx]

        print(f"  Train patients: {len(train_subjects)}  |  Test patients: {len(test_subjects)}")
        print(f"  Train label dist: {dict(zip(*np.unique(train_labels_fold, return_counts=True)))}")
        print(f"  Test  label dist: {dict(zip(*np.unique(test_labels_fold, return_counts=True)))}")

        # ── PATH A: Deep Learning ────────────────────────────

        dl_train_mask = np.array([s in train_subjects for s in dl_subjs])
        dl_test_mask  = np.array([s in test_subjects  for s in dl_subjs])

        X_dl_train = dl_seqs[dl_train_mask].copy()
        X_dl_test  = dl_seqs[dl_test_mask].copy()

        dl_train_subj_arr = np.array(dl_subjs)[dl_train_mask]
        dl_test_subj_arr  = np.array(dl_subjs)[dl_test_mask]

        label_map = dict(zip(patient_df['subject_id'], patient_df[label_col]))
        y_dl_train = np.array([label_map[s] for s in dl_train_subj_arr])
        y_dl_test  = np.array([label_map[s] for s in dl_test_subj_arr])

        # Z-score normalisation (fit on train only)
        n_tr, seq_l, n_f = X_dl_train.shape
        n_te = X_dl_test.shape[0]

        scaler_dl = StandardScaler()
        X_dl_train_flat = X_dl_train.reshape(-1, n_f)
        scaler_dl.fit(X_dl_train_flat)

        X_dl_train_scaled = scaler_dl.transform(X_dl_train_flat).reshape(n_tr, seq_l, n_f)
        X_dl_test_scaled  = scaler_dl.transform(X_dl_test.reshape(-1, n_f)).reshape(n_te, seq_l, n_f)

        print(f"  DL train: {X_dl_train_scaled.shape}  |  DL test: {X_dl_test_scaled.shape}")

        # ── PATH B: Machine Learning ─────────────────────────

        ml_train = ml_feat_df[ml_feat_df['subject_id'].isin(train_subjects)].copy()
        ml_test  = ml_feat_df[ml_feat_df['subject_id'].isin(test_subjects)].copy()

        y_ml_train = ml_train['subject_id'].map(label_map).values.astype(int)
        y_ml_test  = ml_test['subject_id'].map(label_map).values.astype(int)

        scaler_ml = StandardScaler()
        X_ml_train_raw = ml_train[ml_feat_cols].values.astype(np.float64)
        X_ml_test_raw  = ml_test[ml_feat_cols].values.astype(np.float64)

        # Handle any NaN/Inf from feature engineering
        X_ml_train_raw = np.nan_to_num(X_ml_train_raw, nan=0.0, posinf=0.0, neginf=0.0)
        X_ml_test_raw  = np.nan_to_num(X_ml_test_raw,  nan=0.0, posinf=0.0, neginf=0.0)

        scaler_ml.fit(X_ml_train_raw)
        X_ml_train_scaled = scaler_ml.transform(X_ml_train_raw)
        X_ml_test_scaled  = scaler_ml.transform(X_ml_test_raw)

        print(f"  ML train: {X_ml_train_scaled.shape}  |  ML test: {X_ml_test_scaled.shape}")

        # ── Store fold data ──────────────────────────────────
        fold_data = {
            'fold': fold_idx,
            'train_subjects': train_subjects,
            'test_subjects':  test_subjects,

            # Path A – Deep Learning
            'X_dl_train':    X_dl_train_scaled.astype(np.float32),
            'X_dl_test':     X_dl_test_scaled.astype(np.float32),
            'y_dl_train':    y_dl_train,
            'y_dl_test':     y_dl_test,
            'subj_dl_train': dl_train_subj_arr,
            'subj_dl_test':  dl_test_subj_arr,
            'scaler_dl':     scaler_dl,

            # Path B – Machine Learning
            'X_ml_train':    X_ml_train_scaled,
            'X_ml_test':     X_ml_test_scaled,
            'y_ml_train':    y_ml_train,
            'y_ml_test':     y_ml_test,
            'subj_ml_train': ml_train['subject_id'].values,
            'subj_ml_test':  ml_test['subject_id'].values,
            'scaler_ml':     scaler_ml,
            'ml_feat_cols':  ml_feat_cols,
        }

        folds.append(fold_data)

    return folds


# ============================================================
# 17.  RUN PREPROCESSING — TASK 1: DETECTION
#      (Healthy Control vs PD)
# ============================================================
print("\n" + "█"*60)
print("  TASK 1: DETECTION  (HC=0  vs  PD=1)")
print("█"*60)

detection_patient_df = patient_table[['subject_id', 'label_detect']].copy()
detection_subjects   = set(detection_patient_df['subject_id'])

det_folds = run_cv_preprocessing(
    task_name='Detection',
    patient_df=detection_patient_df,
    label_col='label_detect',
    full_data=full_df,
    dl_seqs=dl_sequences,
    dl_subjs=dl_subjects,
    ml_feat_df=ml_features[ml_features['subject_id'].isin(detection_subjects)],
    ml_feat_cols=ml_feat_cols
)

# ============================================================
# 18.  RUN PREPROCESSING — TASK 2: STAGING
#      (Early PD=0  vs  Moderate PD=1)
# ============================================================
print("\n" + "█"*60)
print("  TASK 2: STAGING  (Early PD=0  vs  Moderate PD=1)")
print("█"*60)

staging_patient_df = patient_table_staging[['subject_id', 'label_stage']].copy()
staging_subjects   = set(staging_patient_df['subject_id'])

staging_dl_mask  = np.array([s in staging_subjects for s in dl_subjects])
staging_dl_seqs  = dl_sequences[staging_dl_mask]
staging_dl_subjs = [s for s, m in zip(dl_subjects, staging_dl_mask) if m]

stg_folds = run_cv_preprocessing(
    task_name='Staging',
    patient_df=staging_patient_df,
    label_col='label_stage',
    full_data=full_df[full_df['subject_id'].isin(staging_subjects)],
    dl_seqs=staging_dl_seqs,
    dl_subjs=staging_dl_subjs,
    ml_feat_df=ml_features[ml_features['subject_id'].isin(staging_subjects)],
    ml_feat_cols=ml_feat_cols
)

# ============================================================
# 19.  SUMMARY & VERIFICATION
# ============================================================
print("\n" + "="*60)
print("  PREPROCESSING COMPLETE — SUMMARY")
print("="*60)
print(f"\n  Sequence length used (avg across all drawings): {SEQ_LEN}")

for task_label, fold_list in [("Detection", det_folds), ("Staging", stg_folds)]:
    print(f"\n{'─'*40}")
    print(f"  {task_label}: {len(fold_list)} folds")
    print(f"{'─'*40}")
    for f in fold_list:
        i = f['fold']
        print(f"  Fold {i+1}:")
        print(f"    DL  →  train {f['X_dl_train'].shape}  test {f['X_dl_test'].shape}")
        print(f"    ML  →  train {f['X_ml_train'].shape}  test {f['X_ml_test'].shape}")
        print(f"    y_train dist: {dict(zip(*np.unique(f['y_dl_train'], return_counts=True)))}")
        print(f"    y_test  dist: {dict(zip(*np.unique(f['y_dl_test'], return_counts=True)))}")

        # Verify no patient leakage
        overlap = f['train_subjects'] & f['test_subjects']
        assert len(overlap) == 0, f"DATA LEAKAGE in fold {i+1}! Overlapping: {overlap}"

    print(f"  ✅  No patient leakage detected across all {len(fold_list)} folds.")

print("\n✅  All preprocessing pipelines ready.")
print(f"    • SEQ_LEN = {SEQ_LEN} (computed as mean across {len(seq_lengths)} drawings)")
print("    • det_folds  — list of 5 fold dicts for Detection task")
print("    • stg_folds  — list of 5 fold dicts for Staging task")

Patient table (all):
group
Healthy Control    38
Early PD           23
Moderate PD        11
Name: count, dtype: int64

Total patients: 72

Staging patients: 34
group
Early PD       23
Moderate PD    11
Name: count, dtype: int64

── Sequence Length Statistics ──────────────────────────────
  Total drawings     : 573
  Min  seq length    : 405
  Max  seq length    : 16071
  Mean seq length    : 2275.4
  Median seq length  : 1979.0
  Std  seq length    : 1319.7
  ► SEQ_LEN (rounded): 2275

Building DL sequences (SEQ_LEN = 2275) …
  DL tensor shape : (573, 2275, 4)
Building ML feature vectors …
  ML feature matrix : (573, 28)
  Feature columns   : 24
  Features: ['velocity_mean', 'velocity_median', 'velocity_std', 'velocity_max', 'accel_mean', 'accel_median', 'accel_std', 'accel_max', 'jerk_mean', 'jerk_median', 'jerk_std', 'jerk_max', 'pressure_mean', 'pressure_median', 'pressure_std', 'pressure_max', 'total_in_air_time', 'total_on_surface_time', 'total_time', 'in_air_time_ratio', 'strok

MODEL TRAINING

In [9]:
# ============================================================
# 20.  COMMON IMPORTS & UTILITIES FOR MODEL TRAINING
# ============================================================
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
import json

# ============================================================
# 20a. EVALUATION HELPER (shared by all models)
# ============================================================
def evaluate_predictions(y_true, y_pred, y_prob=None):
    """Compute standard binary classification metrics."""
    metrics = {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
    }
    if y_prob is not None:
        try:
            metrics['auc_roc'] = roc_auc_score(y_true, y_prob)
        except ValueError:
            metrics['auc_roc'] = float('nan')
    else:
        metrics['auc_roc'] = float('nan')
    metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred).tolist()
    return metrics


def print_fold_metrics(metrics, fold_idx):
    print(f"    Accuracy : {metrics['accuracy']:.4f}")
    print(f"    Precision: {metrics['precision']:.4f}")
    print(f"    Recall   : {metrics['recall']:.4f}")
    print(f"    F1       : {metrics['f1']:.4f}")
    print(f"    AUC-ROC  : {metrics['auc_roc']:.4f}")
    print(f"    Confusion: {metrics['confusion_matrix']}")


def summarise_cv(all_metrics, model_name, task_name):
    """Print mean ± std across folds."""
    print(f"\n{'═'*60}")
    print(f"  {model_name} — {task_name} — 5-Fold CV Summary")
    print(f"{'═'*60}")
    for key in ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']:
        vals = [m[key] for m in all_metrics if not np.isnan(m[key])]
        if vals:
            print(f"  {key:<12}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")
        else:
            print(f"  {key:<12}: N/A")
    print(f"{'═'*60}")
    return {
        key: {'mean': float(np.mean([m[key] for m in all_metrics if not np.isnan(m[key])])),
              'std':  float(np.std([m[key] for m in all_metrics if not np.isnan(m[key])]))}
        for key in ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
    }

LSTM

In [11]:
# ============================================================
# 24.  MODEL 4 — LSTM  (Path A: DL sequences)
# ============================================================
# Run this cell independently for LSTM training & evaluation

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ── Check device ─────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("╔" + "═"*58 + "╗")
print("║           MODEL 4: LSTM (DL Sequences)                 ║")
print("╚" + "═"*58 + "╝")

# ── LSTM Architecture ────────────────────────────────────────
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.3):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # Use last hidden state of final layer
        last_hidden = h_n[-1]             # (batch, hidden_dim)
        out = self.dropout(last_hidden)
        out = self.fc(out)                # (batch, 1)
        return out.squeeze(-1)


# ── Training function ────────────────────────────────────────
def train_dl_model(model, train_loader, val_X, val_y,
                   epochs=50, lr=1e-3, patience=10, task_name=''):
    """
    Train a PyTorch model with early stopping on validation loss.
    """
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )

    val_X_t = torch.tensor(val_X, dtype=torch.float32).to(device)
    val_y_t = torch.tensor(val_y, dtype=torch.float32).to(device)

    best_val_loss = float('inf')
    best_state    = None
    no_improve    = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        n_batches  = 0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            logits = model(batch_X)
            loss   = criterion(logits, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            n_batches  += 1

        avg_train_loss = train_loss / max(n_batches, 1)

        # Validation
        model.eval()
        with torch.no_grad():
            val_logits = model(val_X_t)
            val_loss   = criterion(val_logits, val_y_t).item()

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1

        if (epoch + 1) % 10 == 0 or no_improve == 0:
            print(f"      Epoch {epoch+1:>3}/{epochs} — "
                  f"train_loss: {avg_train_loss:.4f}  val_loss: {val_loss:.4f}"
                  f"{'  ★' if no_improve == 0 else ''}")

        if no_improve >= patience:
            print(f"      Early stopping at epoch {epoch+1}")
            break

    # Restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)
    model.to(device)
    return model


def predict_dl_model(model, X):
    """Get predictions and probabilities from a trained PyTorch model."""
    model.eval()
    X_t = torch.tensor(X, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_t)
        probs  = torch.sigmoid(logits).cpu().numpy()
    preds = (probs >= 0.5).astype(int)
    return preds, probs


# ── Run LSTM across both tasks ───────────────────────────────
LSTM_PARAMS = {
    'hidden_dim': 64,
    'num_layers': 2,
    'dropout':    0.3,
    'epochs':     50,
    'lr':         1e-3,
    'batch_size': 32,
    'patience':   10,
}

lstm_results = {}

for task_name, fold_list in [("Detection", det_folds), ("Staging", stg_folds)]:
    print(f"\n{'▓'*60}")
    print(f"  LSTM — {task_name}")
    print(f"{'▓'*60}")

    fold_metrics = []

    for fold_data in fold_list:
        fold_idx = fold_data['fold']
        print(f"\n  ── Fold {fold_idx + 1}/5 ──")

        X_train = fold_data['X_dl_train']
        X_test  = fold_data['X_dl_test']
        y_train = fold_data['y_dl_train'].astype(np.float32)
        y_test  = fold_data['y_dl_test'].astype(np.float32)

        input_dim = X_train.shape[2]  # number of features (4)

        # DataLoader
        train_ds     = TensorDataset(
            torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.float32)
        )
        train_loader = DataLoader(
            train_ds,
            batch_size=LSTM_PARAMS['batch_size'],
            shuffle=True,
            drop_last=False
        )

        # Build model
        model = LSTMClassifier(
            input_dim=input_dim,
            hidden_dim=LSTM_PARAMS['hidden_dim'],
            num_layers=LSTM_PARAMS['num_layers'],
            dropout=LSTM_PARAMS['dropout']
        ).to(device)

        print(f"    Parameters: {sum(p.numel() for p in model.parameters()):,}")

        start = time.time()
        model = train_dl_model(
            model, train_loader, X_test, y_test,
            epochs=LSTM_PARAMS['epochs'],
            lr=LSTM_PARAMS['lr'],
            patience=LSTM_PARAMS['patience'],
            task_name=task_name
        )
        train_time = time.time() - start

        y_pred, y_prob = predict_dl_model(model, X_test)

        metrics = evaluate_predictions(y_test.astype(int), y_pred, y_prob)
        metrics['train_time'] = train_time
        fold_metrics.append(metrics)

        print(f"    Train time: {train_time:.2f}s")
        print_fold_metrics(metrics, fold_idx)

        # Free GPU memory
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    summary = summarise_cv(fold_metrics, "LSTM", task_name)
    lstm_results[task_name] = {
        'fold_metrics': fold_metrics,
        'summary': summary
    }

print("\n✅ LSTM complete. Results stored in `lstm_results`.")

Using device: cpu
╔══════════════════════════════════════════════════════════╗
║           MODEL 4: LSTM (DL Sequences)                 ║
╚══════════════════════════════════════════════════════════╝

▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  LSTM — Detection
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

  ── Fold 1/5 ──
    Parameters: 51,265
      Epoch   1/50 — train_loss: 0.6961  val_loss: 0.6833  ★
      Epoch   3/50 — train_loss: 0.6958  val_loss: 0.6808  ★
      Epoch   5/50 — train_loss: 0.6896  val_loss: 0.6795  ★
      Epoch   6/50 — train_loss: 0.6920  val_loss: 0.6763  ★
      Epoch  10/50 — train_loss: 0.6908  val_loss: 0.6811
      Epoch  11/50 — train_loss: 0.6900  val_loss: 0.6736  ★


KeyboardInterrupt: 

GRU